# Pipeline de Classificação LULC — Sentinel-2 + MapBiomas
**Região:** Sapezal (MT) · **Período:** seca 2023 (Jun–Set) · **Classificador:** Random Forest (GEE) + scikit-learn/XGBoost

Todo processamento de imagem é executado **server-side no Google Earth Engine**.  
O Colab/local lida apenas com tabelas de amostras leves e visualizações.

---
## Etapa 1 — Setup do Ambiente

In [ ]:
import sys
import os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # --- Edite estas variáveis conforme necessário ---
    REPO_URL    = 'https://github.com/ferdamarc/LULC_project.git'
    REPO_BRANCH = 'dev'
    PROJECT_DIR = '/content/LULC_project'
    # -------------------------------------------------

    if not os.path.exists(PROJECT_DIR):
        os.system(f'git clone --branch {REPO_BRANCH} {REPO_URL} {PROJECT_DIR}')
    else:
        os.system(f'git -C {PROJECT_DIR} checkout {REPO_BRANCH}')
        os.system(f'git -C {PROJECT_DIR} pull origin {REPO_BRANCH} --quiet')

    os.chdir(f'{PROJECT_DIR}/notebooks')
    os.system('pip install -r ../requirements.txt -q')
    sys.path.insert(0, f'{PROJECT_DIR}/src')
    print(f'Colab: projeto carregado de {PROJECT_DIR} (branch: {REPO_BRANCH})')

else:
    _cwd = Path(os.getcwd())
    _root = next(
        (p for p in [_cwd] + list(_cwd.parents) if (p / 'environment.yml').exists()),
        _cwd.parent,
    )
    sys.path.insert(0, str(_root / 'src'))
    os.chdir(_root / 'notebooks')
    print(f'Local: usando ambiente Conda lulc-project')
    print(f'Raiz do projeto: {_root}')

In [ ]:
import ee
import geemap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score
import xgboost as xgb
import sys
import os

# Patch permanente para o bug do geemap no Colab: TimeoutException não capturada
# ao buscar GOOGLE_MAPS_API_KEY nos Colab Secrets. Sem efeito fora do Colab.
if IN_COLAB:
    import google.colab.userdata as _ud
    _orig_ud_get = _ud.get
    def _safe_ud_get(key):
        try:
            return _orig_ud_get(key)
        except Exception:
            return None
    _ud.get = _safe_ud_get

from gee_utils import build_s2_composite, load_mapbiomas, FEATURE_BANDS

print(f"ee: {ee.__version__} | geemap: {geemap.__version__}")
print(f"Feature bands ({len(FEATURE_BANDS)}): {FEATURE_BANDS}")

In [ ]:
ee.Authenticate()

GEE_PROJECT = 'lulc-project-499923'

ee.Initialize(project=GEE_PROJECT)
print('GEE inicializado com sucesso.')

---
## Etapa 2 — Definição da AOI e Configuração do Projeto

In [ ]:
# =============================================================================
# Bloco de configuração central — todos os parâmetros do projeto em um lugar.
# Etapas subsequentes leem daqui; não repetir essas constantes nas células abaixo.
# =============================================================================

# --- AOI ---
# Bbox de ~2.400 km² (≈50 km × 50 km) sobre a zona agrícola de Sapezal (MT).
# BBox(west, south, east, north)
AOI = ee.Geometry.BBox(-59.20, -13.85, -58.70, -13.35)

# --- Período temporal ---
START_DATE = '2023-06-01'
END_DATE   = '2023-09-30'

# --- Classes LULC do projeto ---
# Chave: código interno do projeto (1-6)
# 'mapbiomas': lista de códigos MapBiomas Collection 9 que mapeiam para essa classe
# 'color': hex para visualização no mapa
LULC_CLASSES = {
    1: {'name': 'Floresta',                  'mapbiomas': [3],                    'color': '#1f7a1f'},
    2: {'name': 'Cerrado/Savana',            'mapbiomas': [4, 12],                'color': '#d4a028'},
    3: {'name': 'Pastagem',                  'mapbiomas': [15],                   'color': '#b8af4f'},
    4: {'name': 'Agricultura',               'mapbiomas': [18, 21, 39, 40, 41],   'color': '#f5e642'},
    5: {'name': 'Água',                      'mapbiomas': [11, 31, 33],           'color': '#2d6fd3'},
    6: {'name': 'Solo Exposto/Não Vegetado', 'mapbiomas': [24, 25, 30],           'color': '#c49a6c'},
}
# Códigos adicionados em relação à definição inicial:
#   11 (Área Úmida) → Água:       campos alagados às margens do Juruena/Papagaio
#   21 (Mosaico Agropecuário) → Agricultura: mistura de lavoura e pasto, predominante no Cerrado

# Lookup inverso: código MapBiomas → código do projeto (usado na Etapa 5)
MB_TO_LULC = {
    mb_code: proj_code
    for proj_code, info in LULC_CLASSES.items()
    for mb_code in info['mapbiomas']
}

# Listas paralelas para ee.Image.remap (espera listas, não dicts)
MB_FROM = list(MB_TO_LULC.keys())
MB_TO   = list(MB_TO_LULC.values())

print('Configuração carregada.')
print(f'AOI: {AOI.bounds().getInfo()}')
print(f'Período: {START_DATE} → {END_DATE}')
print(f'Classes: {", ".join(v["name"] for v in LULC_CLASSES.values())}')
print(f'Códigos MapBiomas mapeados: {MB_FROM}')

In [ ]:
# Calcula a área da AOI server-side (evita .getInfo() em geometrias complexas)
area_km2 = AOI.area(maxError=100).divide(1e6).getInfo()
print(f'Área da AOI: {area_km2:.0f} km²')

In [ ]:
Map = geemap.Map(center=[-13.60, -58.95], zoom=10)
Map.addLayer(AOI, {'color': 'red', 'fillColor': '00000000'}, 'AOI — Sapezal (MT)')
Map.add_basemap('Esri.WorldImagery')
Map

**Verificação antes de avançar:**
- O contorno vermelho cobre a zona agrícola de Sapezal? (deve mostrar talhões de soja/pastagem no basemap de satélite)
- A área calculada está em ~2.400 km²?
- O GEE não retornou erro de quota ou autenticação?

Se tudo OK, avance para a Etapa 3.

---
## Etapa 3 — Carregamento e Pré-processamento do Sentinel-2

**Coleção:** `COPERNICUS/S2_SR_HARMONIZED` (SR = Surface Reflectance, Harmonized = offset radiométrico entre S2A/S2B corrigido)  
**Período:** Jun–Set 2023 · **Filtro:** `CLOUDY_PIXEL_PERCENTAGE < 20%` · **Composite:** mediana

In [ ]:
s2_composite = build_s2_composite(AOI, START_DATE, END_DATE, max_cloud_pct=20)

# Conta quantas cenas entraram na composição (chamada leve ao GEE)
n_scenes = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(AOI)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .size()
    .getInfo()
)
print(f'Cenas disponíveis para composição: {n_scenes}')
print(f'Bandas no composite: {s2_composite.bandNames().getInfo()}')

In [ ]:
# Visualização do composite — duas composições lado a lado como camadas alternáveis
Map_s2 = geemap.Map(center=[-13.60, -58.95], zoom=11)

# Cores verdadeiras: útil para reconhecimento visual das feições (talhões, rios, mata)
vis_rgb = {'bands': ['B4', 'B3', 'B2'], 'min': 0.0, 'max': 0.25, 'gamma': 1.4}
Map_s2.addLayer(s2_composite, vis_rgb, 'RGB — Cores verdadeiras')

# Falsa cor infravermelha: vegetação ativa aparece em vermelho (NIR no canal R)
# Útil para distinguir Floresta (vermelho vivo) de Pastagem/Solo (tons de rosa/bege)
vis_fci = {'bands': ['B8', 'B4', 'B3'], 'min': 0.0, 'max': 0.45, 'gamma': 1.3}
Map_s2.addLayer(s2_composite, vis_fci, 'Falsa Cor Infravermelha (NIR-R-G)')

Map_s2.addLayer(AOI, {'color': 'white', 'fillColor': '00000000'}, 'AOI')
Map_s2.add_basemap('Esri.WorldImagery')
Map_s2

**Verificação antes de avançar para a Etapa 4:**
- `n_scenes` deve ser ≥ 10 (seca de MT costuma ter 15–30 cenas com <20% nuvem)
- No mapa RGB, talhões agrícolas aparecem em tons de bege/ocre (solo exposto pós-colheita)
- Na falsa cor IR, áreas de floresta aparecem em vermelho intenso; pastagem em rosa claro

Se o mapa aparecer completamente cinza ou com muitos pixels ausentes, reduza `max_cloud_pct` para 30 ou expanda o período para Mai–Out 2023.

---
## Etapa 4 — Índices Espectrais

Os índices já foram calculados dentro de `build_s2_composite` (via `add_spectral_indices`).  
Esta etapa os visualiza e valida os intervalos esperados por classe antes de prosseguir para a amostragem.

In [ ]:
Map_idx = geemap.Map(center=[-13.60, -58.95], zoom=11)

# NDVI: vermelho (baixo) → amarelo → verde escuro (alto)
# Na seca: Floresta ~0.7-0.9 | Cerrado ~0.4-0.7 | Pastagem ~0.3-0.5 | Solo Exposto <0.2
vis_ndvi = {
    'bands': ['NDVI'], 'min': -0.1, 'max': 0.9,
    'palette': ['#d73027', '#f46d43', '#fdae61', '#fee08b', '#d9ef8b', '#a6d96a', '#1a9850'],
}
Map_idx.addLayer(s2_composite, vis_ndvi, 'NDVI')

# NDWI: azul escuro = água (NDWI > 0), bege = terra seca (NDWI < 0)
vis_ndwi = {
    'bands': ['NDWI'], 'min': -0.5, 'max': 0.3,
    'palette': ['#d7191c', '#fdae61', '#ffffbf', '#abd9e9', '#2c7bb6'],
}
Map_idx.addLayer(s2_composite, vis_ndwi, 'NDWI')

Map_idx.addLayer(AOI, {'color': 'white', 'fillColor': '00000000'}, 'AOI')
Map_idx.add_basemap('Esri.WorldImagery')
Map_idx

**Verificação antes de avançar para a Etapa 5:**
- **NDVI:** rios e áreas urbanas aparecem em vermelho (<0.1); talhões colhidos em laranja/amarelo (0.1–0.3); pastagem em amarelo-esverdeado (0.3–0.5); mata em verde escuro (>0.6)
- **NDWI:** calha dos rios Juruena/Papagaio deve aparecer em azul escuro (>0) — boa sanidade check da banda de água
- Se os índices aparecerem fora de escala (tudo branco ou tudo preto), é sinal de que a divisão por 10 000 não foi aplicada — verifique se `mask_s2_clouds` está sendo chamada na coleção

---
## Etapa 5 — MapBiomas: Carregamento e Reclassificação

**Asset:** MapBiomas Collection 9 · **Ano:** 2023 · **Resolução:** 30 m (derivado do Landsat)  
A imagem do MapBiomas é uma `ee.Image` com uma banda por ano. Selecionamos `classification_2023` e remapeamos os códigos originais para as 6 classes do projeto.

In [ ]:
lulc_mapbiomas = load_mapbiomas(AOI, MB_FROM, MB_TO, year=2023)

# Paleta indexada: a cor da posição i corresponde ao valor (min + i)
# min=1 → [Floresta, Cerrado, Pastagem, Agricultura, Água, Solo Exposto]
vis_lulc = {
    'min': 1, 'max': 6,
    'palette': [info['color'] for info in LULC_CLASSES.values()],
}

Map_mb = geemap.Map(center=[-13.60, -58.95], zoom=11)
Map_mb.addLayer(s2_composite,  {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.25, 'gamma': 1.4}, 'S2 RGB')
Map_mb.addLayer(lulc_mapbiomas, vis_lulc, 'MapBiomas 2023 — reclassificado')
Map_mb.addLayer(AOI, {'color': 'white', 'fillColor': '00000000'}, 'AOI')

# Legenda das classes
legend_dict = {info['name']: info['color'] for info in LULC_CLASSES.values()}
Map_mb.add_legend(title='Classes LULC', legend_dict=legend_dict, position='bottomright')

Map_mb

In [ ]:
# Distribuição de área por classe — validação antes de amostrar
# ⚠️ Pode levar 10-30s: reduceRegion sobre ~2.7M pixels a 30m
# Se timeout: aumente scale para 100 ou use bestEffort=True (já ativo)
freq_result = lulc_mapbiomas.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=AOI,
    scale=30,
    maxPixels=1e9,
    bestEffort=True,
).getInfo()

pixel_area_km2 = (30 * 30) / 1e6  # cada pixel = 900 m² = 0.0009 km²

records = [
    {
        'Classe': LULC_CLASSES[int(float(k))]['name'],
        'Área (km²)': round(v * pixel_area_km2, 1),
        'Proporção (%)': None,
        'Cor': LULC_CLASSES[int(float(k))]['color'],
    }
    for k, v in freq_result['lulc'].items()
    if int(float(k)) in LULC_CLASSES
]

df_dist = pd.DataFrame(records).sort_values('Área (km²)', ascending=False).reset_index(drop=True)
total = df_dist['Área (km²)'].sum()
df_dist['Proporção (%)'] = (df_dist['Área (km²)'] / total * 100).round(1)

print(df_dist[['Classe', 'Área (km²)', 'Proporção (%)']].to_string(index=False))
print(f'\nÁrea total mapeada: {total:.0f} km²')

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(df_dist['Classe'], df_dist['Área (km²)'], color=df_dist['Cor'], edgecolor='white')
ax.set_xlabel('Área (km²)')
ax.set_title('Distribuição de classes — MapBiomas 2023, AOI Sapezal (MT)')
ax.invert_yaxis()
plt.tight_layout()

output_path = Path('../outputs/figuras/distribuicao_classes_mapbiomas.png')
output_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(output_path, dpi=150, bbox_inches='tight')
plt.show()

**Verificação antes de avançar para a Etapa 6 (amostragem):**
- Todas as 6 classes devem aparecer na tabela — se alguma estiver ausente, a AOI não contém aquela cobertura e precisamos ajustar o bbox ou fundir classes
- **Distribuição esperada para Sapezal 2023:** Agricultura > Pastagem >> Cerrado/Savana ≈ Floresta > Solo Exposto > Água
- No mapa: alterne entre "S2 RGB" e "MapBiomas reclassificado" para validar visualmente se as cores fazem sentido sobre a imagem de satélite
- Se a figura não foi salva em `outputs/figuras/`: verifique se o diretório existe com `!ls ../outputs/figuras/`
- Se receber `EEException: Asset not found`: o asset ID da Collection 9 mudou — busque "MapBiomas" no [GEE Data Catalog](https://developers.google.com/earth-engine/datasets) e atualize `load_mapbiomas` em `src/gee_utils.py`

---
## Etapa 6 — Amostragem Estratificada
*(implementado na próxima sessão)*

---
## Etapa 7 — Treinamento do Classificador
*(implementado na próxima sessão)*

---
## Etapa 8 — Validação: Matriz de Confusão, Kappa, F1-score
*(implementado na próxima sessão)*

---
## Etapa 9 — Visualização dos Resultados
*(implementado na próxima sessão)*